# Day 16: Data Pipelines, Spark and PySpark
**Medallion layers, Spark DataFrames and joins, Spark SQL, a production-style pipeline and verifying a Codex-generated transformation**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.
* Sections marked **NEW (v4)** were added in program v4.

## 1. Setup
Colab includes Java, so PySpark works after a quick install. In VS Code, install **Java 17** (for example Eclipse Temurin) and then `pip install pyspark`.

In [ ]:
# ===== Option A: Google Colab (active) =====
!pip install -q pyspark openai tiktoken scikit-learn

# ===== Option B: VS Code / local (run in your terminal once) =====
# 1. Install Java 17 and check:  java -version
# 2. pip install pyspark pandas openai tiktoken scikit-learn
import os, sqlite3, time
import pandas as pd
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .master("local[1]")
         .appName("day16")
         .config("spark.sql.shuffle.partitions", "2")   # small data, fewer partitions
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

## 2. Raw data (bronze input)

In [ ]:
%%writefile orders.csv
order_id,customer_id,amount,order_date,city
1001,1,250.50,2026-11-12, Bengaluru
1002,2,1200,2026-11-12,Tokyo
1002,2,1200,2026-11-12,Tokyo
1003,3,,2026-11-12,Osaka
1004,4,-20,2026-11-13,Pune
1005,5,560,2026-11-13,Chennai
1006,6,90.25,2026-11-13,Kyoto
1007,1,3100,2026-11-13,Bengaluru
1008,2,3300,2026-11-12,Tokyo
1009,99,780,2026-11-14,Pune
1010,98,150,2026-11-14,Osaka
1011,3,410,2026-11-14,Osaka

In [ ]:
%%writefile customers.csv
customer_id,name,segment
1,Asha,retail
2,Kenji,enterprise
3,Mei,retail
4,Ravi,retail
5,Priya,enterprise
6,Hana,retail

In [ ]:
orders_raw = spark.read.csv("orders.csv", header=True, inferSchema=True)
customers = spark.read.csv("customers.csv", header=True, inferSchema=True)
orders_raw.printSchema()
orders_raw.show(5)
print("Raw orders:", orders_raw.count(), "| Customers:", customers.count())

## 3. Lazy evaluation in action

In [ ]:
t = time.perf_counter()
plan = (orders_raw.filter(F.col("amount") > 0)
        .withColumn("amount_tax", F.col("amount") * 1.18)
        .select("order_id", "city", "amount_tax"))
print(f"Transformations: {time.perf_counter() - t:.3f} s  (nothing ran yet)")
t = time.perf_counter()
print("Rows:", plan.count())
print(f"Action count(): {time.perf_counter() - t:.3f} s  (the whole plan ran)")

## 4. Silver: clean and enrich

In [ ]:
silver = (orders_raw
    .dropDuplicates(["order_id"])
    .filter(F.col("amount").isNotNull() & (F.col("amount") > 0))
    .withColumn("city", F.trim("city"))
    .withColumn("order_date", F.to_date("order_date"))
    .withColumn("size", F.when(F.col("amount") >= 1000, "large")
                         .when(F.col("amount") >= 300, "medium")
                         .otherwise("small")))
silver.orderBy("order_id").show()
print("Silver rows:", silver.count())

### TODO 1: Add a column `weekday` using `F.date_format("order_date", "E")` and count orders per weekday.

In [ ]:
# TODO: write your code here

## 5. Joins: check counts every time

In [ ]:
inner = silver.join(customers, "customer_id", "inner")
left = silver.join(customers, "customer_id", "left")
unknown = silver.join(customers, "customer_id", "left_anti")

print("silver:", silver.count(), "| inner:", inner.count(), "| left:", left.count())
print("Orders with unknown customer:", [r.order_id for r in unknown.orderBy("order_id").collect()])
left.select("order_id", "name", "segment", "amount").orderBy("order_id").show()

## 6. Gold: aggregate and write the medallion layers

In [ ]:
gold = (left
    .groupBy("order_date", "city")
    .agg(F.count("*").alias("orders"),
         F.round(F.sum("amount"), 2).alias("revenue"),
         F.round(F.avg("amount"), 2).alias("avg_order"))
    .orderBy("order_date", F.desc("revenue")))
gold.show()

orders_raw.write.mode("overwrite").parquet("lake/bronze/orders")
silver.write.mode("overwrite").parquet("lake/silver/orders")
gold.write.mode("overwrite").partitionBy("order_date").parquet("lake/gold/daily_city_revenue")
print(sorted(os.listdir("lake/gold/daily_city_revenue")))
print("Gold rows:", spark.read.parquet("lake/gold/daily_city_revenue").count())

### TODO 2: Re-run the cell above and confirm the gold row count is unchanged. Then try `mode("append")`. Which mode makes the job idempotent?

## 7. SQL refresher in SQLite, then the same query in Spark SQL

In [ ]:
sv = silver.select("order_id", "customer_id", "amount", F.col("order_date").cast("string").alias("order_date"),
                   "city").toPandas()
conn = sqlite3.connect(":memory:")
sv.to_sql("orders", conn, index=False)
customers.toPandas().to_sql("customers", conn, index=False)
q = lambda sql: pd.read_sql(sql, conn)

# Business question: revenue per customer segment, including unknown customers
print(q("""
SELECT COALESCE(c.segment, 'unknown') AS segment, ROUND(SUM(o.amount), 2) AS revenue
FROM orders o LEFT JOIN customers c ON o.customer_id = c.customer_id
GROUP BY COALESCE(c.segment, 'unknown') ORDER BY revenue DESC
"""))

In [ ]:
# Business question: the top city on each day (CTE + window function)
top_city_sql = """
WITH daily AS (
    SELECT order_date, city, ROUND(SUM(amount), 2) AS revenue
    FROM orders
    GROUP BY order_date, city
),
ranked AS (
    SELECT *, RANK() OVER (PARTITION BY order_date ORDER BY revenue DESC) AS rnk
    FROM daily
)
SELECT order_date, city, revenue FROM ranked WHERE rnk = 1 ORDER BY order_date
"""
sqlite_result = q(top_city_sql)

silver.withColumn("order_date", F.col("order_date").cast("string")).createOrReplaceTempView("orders")
spark_result = spark.sql(top_city_sql)
spark_result.show()
same = [tuple(r) for r in spark_result.collect()] == [tuple(r) for r in sqlite_result.itertuples(index=False)]
print("Spark result == SQLite result:", same)

### TODO 3: Write a query with `LAG` that shows each day's total revenue and the change from the previous day.

In [ ]:
# TODO: write your code here

## 8. A production-style pipeline: run date, checks and a run log

In [ ]:
%%writefile pipeline.py
import time
from pyspark.sql import DataFrame, functions as F

class PipelineError(Exception):
    pass

def to_silver(bronze: DataFrame) -> DataFrame:
    return (bronze.dropDuplicates(["order_id"])
            .filter(F.col("amount").isNotNull() & (F.col("amount") > 0))
            .withColumn("city", F.trim("city")))

def check_not_empty(df: DataFrame, name: str):
    if df.count() == 0:
        raise PipelineError(f"{name} is empty")

def check_unique(df: DataFrame, key: str):
    if df.count() != df.select(key).distinct().count():
        raise PipelineError(f"duplicate {key}")

def to_gold(spark, silver: DataFrame, customers: DataFrame) -> DataFrame:
    silver.createOrReplaceTempView("silver")
    customers.createOrReplaceTempView("dim_customers")
    return spark.sql("""
        SELECT s.city, COALESCE(c.segment, 'unknown') AS segment,
               COUNT(*) AS orders, ROUND(SUM(s.amount), 2) AS revenue
        FROM silver s LEFT JOIN dim_customers c ON s.customer_id = c.customer_id
        GROUP BY s.city, COALESCE(c.segment, 'unknown')
    """)

def check_reconciles(silver: DataFrame, gold: DataFrame):
    s = silver.agg(F.round(F.sum("amount"), 2)).first()[0]
    g = gold.agg(F.round(F.sum("revenue"), 2)).first()[0]
    if abs(s - g) > 0.01:
        raise PipelineError(f"gold {g} != silver {s}")

def run(spark, orders: DataFrame, customers: DataFrame, run_date: str, out_path: str, log: list):
    start = time.perf_counter()
    bronze = orders.filter(F.col("order_date") == run_date)
    silver = to_silver(bronze)
    check_not_empty(silver, "silver")
    check_unique(silver, "order_id")
    gold = to_gold(spark, silver, customers)
    check_reconciles(silver, gold)
    gold.write.mode("overwrite").parquet(f"{out_path}/order_date={run_date}")
    entry = {"run_date": run_date, "bronze": bronze.count(), "silver": silver.count(),
             "gold": gold.count(), "seconds": round(time.perf_counter() - start, 2)}
    log.append(entry)
    return entry

In [ ]:
import importlib, sys
sys.path.insert(0, os.getcwd())
import pipeline
importlib.reload(pipeline)
bronze_orders = orders_raw.withColumn("order_date", F.col("order_date").cast("string"))
run_log = []
for d in ["2026-11-12", "2026-11-13", "2026-11-14"]:
    print(pipeline.run(spark, bronze_orders, customers, d, "lake/gold/daily", run_log))
print("Re-run:", pipeline.run(spark, bronze_orders, customers, "2026-11-13", "lake/gold/daily", run_log))
pd.DataFrame(run_log)

## 9. Tests for the quality gates

In [ ]:
%%writefile test_pipeline.py
import pytest
from pyspark.sql import SparkSession
import pipeline

@pytest.fixture(scope="session")
def spark():
    s = SparkSession.builder.master("local[1]").appName("tests").getOrCreate()
    yield s

def test_silver_removes_bad_and_duplicate_rows(spark):
    df = spark.createDataFrame([(1, 1, 10.0, "d", " A "), (1, 1, 10.0, "d", " A "), (2, 1, -5.0, "d", "B")],
                               ["order_id", "customer_id", "amount", "order_date", "city"])
    out = pipeline.to_silver(df).collect()
    assert len(out) == 1 and out[0].city == "A"

def test_empty_raises(spark):
    df = spark.createDataFrame([], "order_id INT")
    with pytest.raises(pipeline.PipelineError):
        pipeline.check_not_empty(df, "silver")

def test_duplicates_raise(spark):
    df = spark.createDataFrame([(1,), (1,)], ["order_id"])
    with pytest.raises(pipeline.PipelineError):
        pipeline.check_unique(df, "order_id")

In [ ]:
# ===== Option A: Google Colab (active) =====
!python -m pytest -q test_pipeline.py

# ===== Option B: VS Code =====
# Terminal: pytest -q test_pipeline.py

## 10. NEW (v4): AI-assisted data engineering: generate a transformation with Codex and verify it
We ask for `sla_metrics(tickets)` (SmartDesk step 14). Online, the cell asks the model. Offline, it loads a saved AI draft. Then we verify it against a **golden example** with a hand-calculated answer.

## Setup and API key
With an `OPENAI_API_KEY` the notebook calls a real model. Without one, it runs in **offline demo mode** using a tiny stand-in, so every cell still works and you can study the code. Offline scores show how the code works, not real model quality.

The model name comes from the `LLM_MODEL` environment variable. The default is the program's approved default; confirm it against the approved model list in `docs/MODEL_GUIDE.md`.

In [ ]:
%%writefile genai_helpers.py
"""Shared helpers for the GenAI days.
llm()   -> calls OpenAI when a key is available, otherwise a small OFFLINE stand-in so every cell still runs.
embed() -> OpenAI embeddings online, a hashing-based stand-in offline (same API, lower quality).
usage() -> token counts of the last llm() call (estimated offline).
"""
import json, os, re
import numpy as np

def _get_key():
    if os.getenv("OFFLINE") == "1":                 # force offline demo mode (used by tools/check_notebooks.py)
        return None
    try:
        from google.colab import userdata          # Colab Secrets
        return userdata.get("OPENAI_API_KEY")
    except Exception:
        return os.getenv("OPENAI_API_KEY")

OPENAI_API_KEY = _get_key()
ONLINE = bool(OPENAI_API_KEY)
# Default model: confirm against the approved model list (docs/MODEL_GUIDE.md). Override with LLM_MODEL.
MODEL = os.getenv("LLM_MODEL", "gpt-6.1-sol")
EMBED_MODEL = os.getenv("EMBED_MODEL", "text-embedding-3-small")
if ONLINE:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

WORD = re.compile(r"[a-z0-9]+")
LAST_USAGE = {"input_tokens": 0, "output_tokens": 0}

_ENC = []

def count_tokens(text):
    if not _ENC:
        try:
            import tiktoken
            _ENC.append(tiktoken.get_encoding("o200k_base"))
        except Exception:
            _ENC.append(None)                        # no tiktoken or no download: use the estimate
    if _ENC[0] is not None:
        return len(_ENC[0].encode(text))
    return max(1, round(len(text) / 4))              # rough rule: about 4 characters per token

def _words(t):
    # crude normalisation so "expires" matches "expire"
    return {w[:-1] if w.endswith("s") and len(w) > 4 else w for w in WORD.findall(t.lower()) if len(w) > 3}

def offline_llm(prompt, system=""):
    """A deliberately simple stand-in. It is NOT intelligent; it lets the code paths run without a key."""
    low = prompt.lower()
    if "context:" in low and "question:" in low:
        ctx = re.split(r"(?i)context:", prompt, 1)[1]
        ctx, q = re.split(r"(?i)question:", ctx, 1)
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", ctx.strip()) if s.strip()]
        best = max(sents, key=lambda s: len(_words(s) & _words(q)), default="")
        return best if best and _words(best) & _words(q) else "I don't know based on the provided context."
    if "classify" in low and "positive" in low:
        neg = sum(w in low for w in ["slow", "bad", "broken", "late", "angry", "worst", "refund"])
        pos = sum(w in low for w in ["great", "love", "fast", "thanks", "excellent", "happy"])
        return "negative" if neg > pos else "positive" if pos > neg else "neutral"
    if "classify" in low:                            # SmartDesk ticket categories
        last = low.strip().splitlines()[-1]
        for label, words in [("billing", ["bill", "charge", "refund", "invoice", "payment"]),
                             ("network", ["signal", "network", "slow", "outage", "data", "5g"]),
                             ("account", ["password", "login", "account", "sim", "plan"])]:
            if any(w in last for w in words):
                return label
        return "other"
    if "json" in low:
        text = prompt.strip().splitlines()[-1]
        return json.dumps({"summary": " ".join(text.split()[:12]), "offline": True})
    first = re.split(r"(?<=[.!?])\s+", prompt.strip())[0]
    return f"[offline demo] {first[:160]}"

def llm(prompt, system="You are a helpful assistant.", temperature=0.2, model=None, max_tokens=None):
    if ONLINE:
        kwargs = {"max_tokens": max_tokens} if max_tokens else {}
        r = client.chat.completions.create(
            model=model or MODEL, temperature=temperature,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}], **kwargs)
        LAST_USAGE.update(input_tokens=r.usage.prompt_tokens, output_tokens=r.usage.completion_tokens)
        return r.choices[0].message.content
    out = offline_llm(prompt, system)
    LAST_USAGE.update(input_tokens=count_tokens(system + prompt), output_tokens=count_tokens(out))
    return out

def usage():
    return dict(LAST_USAGE)

def embed(texts):
    """Return an (n, d) array of unit-length vectors."""
    if isinstance(texts, str):
        texts = [texts]
    if ONLINE:
        r = client.embeddings.create(model=EMBED_MODEL, input=texts)
        v = np.array([d.embedding for d in r.data])
    else:
        from sklearn.feature_extraction.text import HashingVectorizer
        hv = HashingVectorizer(n_features=1024, ngram_range=(1, 2), alternate_sign=False, norm=None)
        v = hv.transform([t.lower() for t in texts]).toarray()
    n = np.linalg.norm(v, axis=1, keepdims=True)
    return v / np.where(n == 0, 1, n)

print("Mode:", f"ONLINE ({MODEL})" if ONLINE else "OFFLINE demo (add OPENAI_API_KEY to use a real model)")

In [ ]:
import importlib, genai_helpers
importlib.reload(genai_helpers)
from genai_helpers import llm, embed, usage, count_tokens, ONLINE, MODEL

In [ ]:
SPEC = """Write a PySpark function sla_metrics(tickets) for SmartDesk.
Input columns: ticket_id, priority (low|medium|high), opened_at, resolved_at (timestamp, may be null).
Rules: SLA minutes are high=60, medium=240, low=1440. Unresolved tickets are excluded.
Duplicate ticket_id rows count once.
Output per priority: tickets, met (resolved within SLA), pct_met rounded to 1 decimal.
Use only pyspark.sql.functions as F. No UDFs. Return only Python code defining sla_metrics."""

SAVED_AI_DRAFT = """from pyspark.sql import functions as F

def sla_metrics(tickets):
    sla = (F.when(F.col("priority") == "high", 60)
            .when(F.col("priority") == "medium", 240)
            .otherwise(1440))
    minutes = (F.col("resolved_at").cast("long") - F.col("opened_at").cast("long")) / 60
    return (tickets
            .filter(F.col("resolved_at").isNotNull())
            .withColumn("met", (minutes < sla).cast("int"))
            .groupBy("priority")
            .agg(F.count("*").alias("tickets"), F.sum("met").alias("met"))
            .withColumn("pct_met", F.round(F.col("met") / F.col("tickets") * 100, 1)))
"""

draft = llm(SPEC) if ONLINE else SAVED_AI_DRAFT
draft = draft.strip().removeprefix("```python").removesuffix("```").strip()
print(f"Prompt tokens: {count_tokens(SPEC)} | source: {'live model' if ONLINE else 'saved offline draft'}\n")
print(draft)
ns = {}
exec(draft, ns)            # we only run code we have read, in a notebook, on toy data
sla_metrics_ai = ns["sla_metrics"]

### Golden example
Six rows, answer worked out by hand **before** running anything:

| ticket | priority | minutes to resolve | met? |
|---|---|---|---|
| T1 | high | 30 | yes |
| T2 | high | 90 | no |
| T2 (duplicate row) | high | 90 | counts once |
| T3 | medium | 240 | yes (exactly on the SLA counts as met) |
| T4 | medium | 100 | yes |
| T5 | low | not resolved | excluded |

Expected: high 2 tickets, 1 met, 50.0%; medium 2 tickets, 2 met, 100.0%; no low row.

In [ ]:
from datetime import datetime, timedelta
t0 = datetime(2026, 11, 17, 9, 0)
rows = [("T1", "high", t0, t0 + timedelta(minutes=30)), ("T2", "high", t0, t0 + timedelta(minutes=90)),
        ("T2", "high", t0, t0 + timedelta(minutes=90)), ("T3", "medium", t0, t0 + timedelta(minutes=240)),
        ("T4", "medium", t0, t0 + timedelta(minutes=100)), ("T5", "low", t0, None)]
golden = spark.createDataFrame(rows, "ticket_id string, priority string, opened_at timestamp, resolved_at timestamp")
expected = {"high": (2, 1, 50.0), "medium": (2, 2, 100.0)}

def check(fn, name):
    got = {r.priority: (r.tickets, r.met, r.pct_met) for r in fn(golden).collect()}
    ok = got == expected
    print(f"{name:<10} {'PASS' if ok else 'FAIL'}  got={got}")
    return ok

_ = check(sla_metrics_ai, "AI draft")

### Review notes (offline draft)
* **Duplicates:** no `dropDuplicates("ticket_id")`, so T2 counts twice.
* **Boundary:** uses `<` instead of `<=`, so T3 (exactly 240 minutes) is marked as missed.
* Everything else (null filter, SLA table, rounding) matches the rules.

Verdict: **fix**. The reviewed version below changes only those two lines.

In [ ]:
def sla_metrics(tickets):
    sla = (F.when(F.col("priority") == "high", 60)
            .when(F.col("priority") == "medium", 240)
            .otherwise(1440))
    minutes = (F.col("resolved_at").cast("long") - F.col("opened_at").cast("long")) / 60
    return (tickets
            .dropDuplicates(["ticket_id"])                         # fix 1: count each ticket once
            .filter(F.col("resolved_at").isNotNull())
            .withColumn("met", (minutes <= sla).cast("int"))       # fix 2: on the SLA counts as met
            .groupBy("priority")
            .agg(F.count("*").alias("tickets"), F.sum("met").alias("met"))
            .withColumn("pct_met", F.round(F.col("met") / F.col("tickets") * 100, 1)))

_ = check(sla_metrics, "reviewed")

# Second method: the same number with pandas
pdf = golden.toPandas().drop_duplicates("ticket_id").dropna(subset=["resolved_at"])
pdf["mins"] = (pdf.resolved_at - pdf.opened_at).dt.total_seconds() / 60
pdf["met"] = pdf.mins <= pdf.priority.map({"high": 60, "medium": 240, "low": 1440})
print(pdf.groupby("priority").met.mean().mul(100).round(1).to_dict())

### TODO 4 (SmartDesk step 14): add a `day` column (date of `opened_at`) so gold shows SLA met % per priority per day, write it as Parquet partitioned by day, and extend the golden example to two days.

In [ ]:
# TODO: write your code here

---
**Ganbatte (頑張って): *Do your best***

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)